# What to do with a result file

**A predicted expression matrix is not the output you wanted. A picture of
where a gene sits on the tissue is.** This notebook takes the `.h5ad` from
[01-first-prediction.ipynb](01-first-prediction.ipynb) and does three things
with it:

1. **a spatial gene expression map** — named genes drawn over the tissue;
2. **a UMAP** of the predicted expression;
3. **clustering** of spots, drawn back on the slide.

It ends by putting the prediction beside this section's own measured assay.
**That is a picture to look at, not a score.** No metric is computed anywhere
in this notebook, and no statement about accuracy should be read out of it.
A prediction is a hypothesis about what an assay would have measured.

The result arrives as an `AnnData`, so everything here is ordinary
`scanpy`/`anndata` work. The only part that is specific to this service is the
gene axis, and that is the first section, because it is where a first analysis
usually goes wrong.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import anndata
import h5py
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
PREDICTION = DATA / "prediction.h5ad"

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=110, facecolor="white", frameon=False)

if not PREDICTION.exists():
    raise SystemExit(
        f"{PREDICTION} does not exist. Run 01-first-prediction.ipynb first: "
        "it is what produces this file."
    )

adata = anndata.read_h5ad(PREDICTION)
print(adata)

In [ ]:
print("spots:", adata.n_obs)
print("genes:", adata.n_vars)
print("coordinates:", adata.obsm["spatial"].shape,
      "-- the tile's centre, in full-resolution slide pixels")
print("model:", adata.uns["model"]["id"])

## 1. The gene axis, and the mistake to avoid

**The rows of `var` are indexed by Ensembl stable gene identifier. The symbol
travels beside the index, in `var["feature_name"]`.**

That is deliberate and it matters. Gene symbols are renamed, retired and
occasionally reassigned to a different gene; identifiers are not. Code that
indexes a result by symbol works until the day a symbol moves, and then it
quietly returns a different gene's column without raising anything.

So: turn the names you hold into identifiers once, then work in identifiers
and use the symbol only for labels.

In [ ]:
symbol_to_id = {
    symbol: gene_id
    for gene_id, symbol in zip(adata.var_names, adata.var["feature_name"])
}
id_to_symbol = dict(zip(adata.var_names, adata.var["feature_name"]))

print("index is:", adata.var_names.name or "(unnamed)", "->", adata.var_names[0])
print("symbol  :", adata.var['feature_name'].iloc[0])
print()
print("MS4A1 is column", adata.var_names.get_loc(symbol_to_id["MS4A1"]),
      "under", symbol_to_id["MS4A1"])

If a symbol you hold is not in the map above, do not guess. Previous symbols
and recorded aliases resolve through the service's own table:
`client.resolve_genes([...])`, or `resolve_genes` from the command line and
from an agent. A name that names two genes is refused rather than guessed at,
which is the behaviour you want — `OI4` is a recorded alias of two different
genes.

## 2. The genes, and why these ones

This section is **lung cancer with tertiary lymphoid structures**. A tertiary
lymphoid structure is an organised lymphoid aggregate that forms in chronically
inflamed or tumour-bearing tissue: a B-cell follicle with a T-cell zone beside
it. On H&E it is a dense, dark, round collection of small cells inside the
tumour bed, and it is visible by eye.

So the panel below is six textbook lineage markers, each with an unambiguous
compartment, chosen so that **a reader can check the picture against the
stained section themselves**:

| Gene | Compartment | Why it is in the panel |
|---|---|---|
| `MS4A1` | B cells | The defining cell of the follicle at the centre of a tertiary lymphoid structure. |
| `CD79A` | B cells | A second, independent B-cell marker. Two markers of one compartment should agree with each other. |
| `CD3E` | T cells | The T-cell zone that sits beside the follicle. |
| `PTPRC` | All leukocytes | Immune infiltrate wherever it is, organised into a structure or not. |
| `COL1A1` | Fibrillar collagen | The stromal scaffold. A structural contrast to the immune genes, and a different part of the tissue. |
| `EPCAM` | Epithelium | The carcinoma compartment. |

A panel of ambiguous genes would make a prettier picture and a worse check.
These six were picked because their expected locations are separable by eye on
the section itself, which is the only thing a map like this can be judged
against without running an assay.

The next cell checks two things rather than assuming them: that every gene is
on the served axis, and that the model saw each one measured in training.

In [ ]:
PANEL = ["MS4A1", "CD79A", "CD3E", "PTPRC", "COL1A1", "EPCAM"]

missing = [s for s in PANEL if s not in symbol_to_id]
if missing:
    raise SystemExit(f"not on this model's gene axis: {missing}")

panel_ids = [symbol_to_id[s] for s in PANEL]
seen = adata.var.loc[panel_ids, "measured_in_training"].astype(bool)

for symbol, gene_id in zip(PANEL, panel_ids):
    print(f"  {symbol:<8} {gene_id}  measured_in_training="
          f"{bool(adata.var.loc[gene_id, 'measured_in_training'])}")

if not seen.all():
    print("\nAt least one gene above was never seen measured. Its column still "
          "reads like any other; the flag is what tells them apart.")

## 3. A spatial gene expression map

`obsm["spatial"]` holds the centre of the tile each row was read from, in
full-resolution slide pixels — the same frame the slide is in, so a map drawn
on it can be laid over the slide.

Two things about the drawing, neither of them cosmetic:

- **Image row numbers increase downwards**, so the vertical axis is inverted.
  Forget this and the map is a mirror image of the tissue, which is a mistake
  nothing in the data will report.
- **The values are on the scale the model's card states**, which notebook 01
  printed as `transform`. The colour range below is clipped at a high
  percentile so that a handful of extreme spots cannot flatten everything
  else; that is a choice about the picture, and it is stated here rather than
  hidden in a helper.

In [ ]:
def draw(values, title, ax, cmap="magma", clip=(1, 99), size=6):
    """One gene, or any per-spot value, over the tissue."""
    xy = adata.obsm["spatial"]
    values = np.asarray(values, dtype=float)
    finite = values[np.isfinite(values)]
    lo, hi = np.percentile(finite, clip) if finite.size else (0, 1)
    handle = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=size, cmap=cmap,
                        vmin=lo, vmax=hi, linewidths=0)
    ax.set_title(title, fontsize=10)
    ax.set_aspect("equal")
    ax.invert_yaxis()        # image rows increase downwards
    ax.set_axis_off()
    return handle

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9.5))
for ax, symbol, gene_id in zip(axes.ravel(), PANEL, panel_ids):
    values = np.asarray(adata[:, gene_id].X).ravel()
    handle = draw(values, f"{symbol}\n{gene_id}", ax)
    fig.colorbar(handle, ax=ax, fraction=0.046, shrink=0.8)
fig.suptitle("Predicted spatial gene expression", y=0.98)
fig.tight_layout()
plt.show()

Read these against each other rather than in isolation. `MS4A1` and `CD79A`
mark one compartment through two different transcripts, so where they agree
you are looking at something the model puts in the same place twice; where
they disagree you are looking at a limit of the prediction. `COL1A1` and
`EPCAM` should be somewhere else entirely — if every panel gene lights up the
same region, the map is telling you about tissue density rather than about
lineage.

Then open the section itself and look. That is the check this panel was chosen
to make possible.

## 4. Prepare the matrix for a neighbourhood analysis

Before a UMAP or a clustering, two columns have to go, and both for the same
reason: a column that is not a prediction must not be averaged with the ones
that are.

- **Genes the model never saw measured.** `var["measured_in_training"]` states
  this per gene. Such a column still holds numbers, and they will cluster and
  colour exactly like any other.
- **Columns that are not finite.** A result may carry a gene as `nan` rather
  than as a number, which is deliberate: zero is a measurement and would be
  averaged, correlated and coloured as one.

What is left goes through the ordinary scanpy reduction: pick the most
variable genes, take principal components of those, build a neighbourhood
graph, embed it.

**The values are not transformed on the way in.** The card states the scale
they arrive on, which notebook 01 printed as `transform`; putting a second
transform on top of it would be taking the same step twice. The one choice
made here is to reduce on the most variable genes rather than on all of them,
which is stated in the call rather than buried in a default.

In [ ]:
keep = adata.var["measured_in_training"].astype(bool).to_numpy()
print(f"{keep.sum():,} genes measured in training, {(~keep).sum():,} not")

work = adata[:, keep].copy()

finite = np.isfinite(np.asarray(work.X)).all(axis=0)
if not finite.all():
    print(f"{(~finite).sum():,} further columns are not finite and are dropped")
    work = work[:, finite].copy()

print("working matrix:", work.shape)

In [ ]:
sc.pp.highly_variable_genes(work, n_top_genes=2000)
print(f"{int(work.var['highly_variable'].sum()):,} genes carry the reduction")

sc.tl.pca(work, n_comps=50, mask_var="highly_variable")
sc.pp.neighbors(work, n_neighbors=15, n_pcs=30)
sc.tl.umap(work)
print("UMAP:", work.obsm["X_umap"].shape)

## 5. The UMAP

A UMAP of the predicted expression: each point is one spot, placed by how
similar its whole predicted transcriptome is to the others. Colouring it by a
panel gene is how you find out whether the structure in the embedding means
anything you can name.

In [ ]:
sc.pl.umap(
    work,
    color=panel_ids,
    title=[f"{s}" for s in PANEL],
    cmap="magma",
    ncols=3,
    frameon=False,
    show=True,
)

## 6. Clustering, and where the clusters are

A UMAP says which spots are alike. It does not say where they are. Put the
cluster labels back on the slide and the two questions are answered at once.

This is the step that turns a matrix into something a pathologist can look at
with you: a cluster that is spatially coherent — a compact region rather than
a scatter of points across the whole section — is a cluster worth naming.

In [ ]:
sc.tl.leiden(work, resolution=0.6, flavor="igraph", n_iterations=2, directed=False)
print(work.obs["leiden"].value_counts().sort_index().to_string())

In [ ]:
sc.pl.umap(work, color="leiden", legend_loc="on data", frameon=False,
           title="Leiden clusters of predicted expression", show=True)

In [ ]:
clusters = work.obs["leiden"]
palette = plt.get_cmap("tab20")
xy = work.obsm["spatial"]

fig, ax = plt.subplots(figsize=(7.5, 7.5))
for index, label in enumerate(clusters.cat.categories):
    at = (clusters == label).to_numpy()
    ax.scatter(xy[at, 0], xy[at, 1], s=6, color=palette(index % 20), label=label,
               linewidths=0)
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_axis_off()
ax.set_title("The same clusters, on the slide")
ax.legend(markerscale=3, fontsize=8, loc="center left", bbox_to_anchor=(1, 0.5),
          frameon=False, title="leiden")
fig.tight_layout()
plt.show()

To find out what a cluster is, ask which genes separate it. `rank_genes_groups`
is the usual scanpy call. It runs on the values as they arrived -- nothing in
this notebook rescaled them -- and the identifiers it returns are translated
back to symbols for reading.

In [ ]:
sc.tl.rank_genes_groups(work, "leiden", method="wilcoxon")

for label in work.obs["leiden"].cat.categories[:4]:
    top = [id_to_symbol.get(g, g)
           for g in work.uns["rank_genes_groups"]["names"][label][:8]]
    print(f"cluster {label}: {', '.join(top)}")

## 7. Beside the measured assay

This section has its own measured spatial expression, which notebook 01
downloaded. Putting the two side by side is worth doing, and it is worth being
exact about what it is.

**It is a picture, not a benchmark.** The two panels are different quantities:
on the left, values on the scale the model's card states; on the right, counts
from one run of an assay on one section, which for a lineage marker means a
small number of transcripts per spot and many spots at zero. They are on
different scales, over different gene sets, with different noise. Nothing here
computes a correlation, and a correlation computed from these two panels would
not mean what it appears to mean.

What the comparison is good for is a coarse question with an honest answer:
**does the predicted signal land in the same part of the tissue as the measured
signal?** Look at where, not at how much.

In [ ]:
MEASURED = DATA / "LC1.h5ad"

if not MEASURED.exists():
    print(f"{MEASURED} is not here. Run section 2 of notebook 01, or:")
    print("  python -m quickstart.zenodo measured")
else:
    with h5py.File(MEASURED) as handle:
        measured_ids = [g.decode() for g in handle["var/gene_ids"][:]]
        # This dataset's own full-resolution columns: x_pixel is the ROW,
        # y_pixel is the COLUMN.
        measured_xy = np.stack([handle["obs/y_pixel"][:],
                                handle["obs/x_pixel"][:]], axis=1)

    # Match on POSITION, not on order: quality control dropped some tiles, so
    # the two files do not have the same rows, and a tile centre need not fall
    # exactly on a measured centre at all -- it will not if you tiled
    # regularly. So: nearest measured spot, accepted only if it is close
    # enough to be the same place.
    from scipy.spatial import cKDTree

    tree = cKDTree(measured_xy)
    pitch_px = float(np.median(tree.query(measured_xy, k=2)[0][:, 1]))
    tolerance = 0.25 * pitch_px   # a quarter of the assay's own spot spacing

    distance, rows = tree.query(adata.obsm["spatial"], k=1)
    matched = distance <= tolerance

    print(f"spot spacing in this assay: {pitch_px:.1f} px")
    print(f"accepting a match within {tolerance:.1f} px of it")
    print(f"{matched.sum():,} of {adata.n_obs:,} predicted spots have a "
          f"measured spot at the same place")

In [ ]:
if MEASURED.exists() and matched.any():
    # Of the panel, show the gene with the most measured signal on this
    # section: a measured panel that is almost entirely zero is not a picture
    # of anything.
    with h5py.File(MEASURED) as handle:
        depth = {}
        columns = {}
        for symbol, gene_id in zip(PANEL, panel_ids):
            if gene_id not in measured_ids:
                continue
            column = handle["X"][:, measured_ids.index(gene_id)]
            columns[symbol] = column
            depth[symbol] = int((column > 0).sum())

    for symbol, count in sorted(depth.items(), key=lambda kv: -kv[1]):
        print(f"  {symbol:<8} measured above zero at {count:,} of "
              f"{len(measured_xy):,} spots")

    shown = max(depth, key=depth.get)
    print(f"\nshowing {shown}")

In [ ]:
if MEASURED.exists() and matched.any():
    gene_id = symbol_to_id[shown]
    predicted = np.asarray(adata[:, gene_id].X).ravel()[matched]
    observed = columns[shown][rows[matched]]
    xy = adata.obsm["spatial"][matched]

    fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
    for ax, values, title, bar, top in (
        # The predicted panel is clipped at a high percentile so a few extreme
        # spots cannot flatten it. The measured panel is not: these are small
        # integer counts with no long tail to clip, and clipping them would
        # hide the handful of spots that carry the signal.
        (axes[0], predicted, f"{shown}: predicted", "model value",
         float(np.percentile(predicted, 99))),
        (axes[1], observed, f"{shown}: measured", "counts",
         float(np.max(observed))),
    ):
        lo = float(np.min(values))
        handle = ax.scatter(xy[:, 0], xy[:, 1], c=values, s=6, cmap="magma",
                            vmin=lo, vmax=max(top, lo + 1e-9), linewidths=0)
        ax.set_aspect("equal")
        ax.invert_yaxis()
        ax.set_axis_off()
        ax.set_title(title, fontsize=11)
        fig.colorbar(handle, ax=ax, fraction=0.046, shrink=0.8, label=bar)

    fig.suptitle("Two different quantities, on the same coordinates. "
                 "Look at where, not at how much.", y=0.97, fontsize=10)
    fig.tight_layout()
    plt.show()

## Next

- **Your own slide.** Notebook 01 runs on any slide the package can open.
  Change the path, let it tile regularly, and the rest is the same.
- **A bulk RNA profile.** This model accepts one as a covariate, it costs
  nothing against your quota, and
  [Add a bulk RNA profile](https://docs.auroraomics.org/guides/bulk-rna/) is
  how to prepare one.
- **A whole slide.**
  [One slide, end to end](https://docs.auroraomics.org/examples/whole-slide/)
  walks a full section through, with the counts at each step.
- **What a result may be used for.**
  [Responsible use](https://docs.auroraomics.org/responsible-use/). A result
  is a hypothesis: it picks the slide, the region and the genes worth the cost
  of measuring.